# Harbour Terminal Teacher Solution

This notebook implements the teacher solution for the BIP harbour terminal scenario.

System summary:
- Intake station + conveyor + camera scan
- Accept/reject based on randomized cargo manifest
- Crane moves accepted containers to ship or temporary wharf storage
- Ship and wharf are both 2 x 3 grids
- Slot accessibility rule: row 2 can only be filled if row 1 below is filled
- Ship roll and draft are updated from the ship simulation model and persisted to SQLite

You can add project photos or diagrams directly in markdown cells, for example:

![Harbour layout](./images/harbour-layout.png)

## 1. Setup and Imports

In [1]:
from pathlib import Path
import sys
from pprint import pprint

ROOT = Path.cwd().parents[1]
SRC = ROOT / "src"
EXAMPLES = ROOT / "examples"
SOLUTION_DIR = EXAMPLES / "bip_teacher_solution_main"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
if str(SOLUTION_DIR) not in sys.path:
    sys.path.insert(0, str(SOLUTION_DIR))

from teacher_solution_lib import build_solution

## 2. Initialize Database + Randomized Manifest

Clarified parameters encoded in the solution module:
- Request topic format: `{base_topic}/{id}/req/{response_id}/{cmd}`
- Base timeout: 10 seconds
- Retries: 1
- Crane positions (mm): pickup x=0, pickup h=86, travel h=300, drop h row1=35, row2=57
- Ship positions (mm): 360, 400, 440
- Wharf positions (mm): 200, 240, 280

In [2]:
# Runtime toggle and connection settings
DRY_RUN = False          # True = simulated conveyor/camera/crane, False = real MQTT services
SYSTEM_ID = "crane-pi-1"  # or "crane-pi-2"
SEED = 7                # set None for non-deterministic manifest
MQTT_HOST = "ed1fe6fe.ala.eu-central-1.emqxsl.com"
MQTT_PORT = 8883
MQTT_USERNAME = "joost-mertens"
MQTT_PASSWORD = "bip-mqtt-lab-2026"
CA_CERT_PATH = "./emqxsl-ca.crt"
print(f"DRY_RUN={DRY_RUN}, SYSTEM_ID={SYSTEM_ID}, SEED={SEED}")

DRY_RUN=False, SYSTEM_ID=crane-pi-1, SEED=7


In [3]:
solution = build_solution(
    db_path=str(SOLUTION_DIR / "harbour-terminal-solution.sqlite"),
    mqtt_host=MQTT_HOST,
    mqtt_port=MQTT_PORT,
    mqtt_base_topic="bip/mqtt-lab",
    system_id=SYSTEM_ID,
    mqtt_username=MQTT_USERNAME,
    mqtt_password=MQTT_PASSWORD,
    mqtt_ca_cert_path=CA_CERT_PATH,
    ship_id=1,
    seed=SEED,
    dry_run=DRY_RUN,
)

print("Randomized cargo manifest (6 containers):")
pprint(solution.manifest_rows())

Randomized cargo manifest (6 containers):
[{'col': 1, 'container_id': 4, 'row': 1, 'slot_id': 1, 'weight_kg': 6516.0},
 {'col': 2, 'container_id': 2, 'row': 1, 'slot_id': 2, 'weight_kg': 11586.0},
 {'col': 3, 'container_id': 12, 'row': 1, 'slot_id': 3, 'weight_kg': 17909.0},
 {'col': 1, 'container_id': 20, 'row': 2, 'slot_id': 4, 'weight_kg': 17402.0},
 {'col': 2, 'container_id': 10, 'row': 2, 'slot_id': 5, 'weight_kg': 4928.0},
 {'col': 3, 'container_id': 1, 'row': 2, 'slot_id': 6, 'weight_kg': 5989.0}]


## 3. Connect to MQTT Services

Make sure these services are running before connecting:
- Conveyor MQTT bridge
- ArUco MQTT detector
- MQTT gantry controller

In [4]:
solution.connect()
print("Connected to MQTT request-response channels.")

Connected to MQTT request-response channels.


## 4. Process One Intake Cycle

Operation flow per cycle:
1. Poll conveyor with `G4`
2. Move to camera using `G2`
3. Scan with `aruco-id`
4. Reject unknown containers, accept known containers
5. Place on ship if target slot is accessible, otherwise place on wharf
6. Cascade from wharf to ship while new slots become accessible

In [ ]:
import time

def process_once(solution):
    # 1) poll conveyor for waiting container
    state = solution.conveyor_state()
    has_container = int(state.get("END_SENSOR", 0)) == 0
    if not has_container:
        return "idle-no-container"

    # 2) move to camera scan position
    solution.conveyor_move_duration(
        direction=solution.config.camera_move_direction,
        duration_ms=solution.config.camera_move_duration_ms,
    )
    time.sleep(solution.config.camera_move_duration_ms/1000)

    # 3) scan with one retry
    scanned_id = None
    for attempt in range(2):
        scan_payload = solution.camera_scan()
        ids = scan_payload.get("id", [])
        if ids:
            scanned_id = int(ids[0])
            break
        if attempt == 0:
            print("No ID detected. Please reposition container and wait for rescan. Retrying in 10 seconds")
            time.sleep(10)

    if scanned_id is None:
        return "scan-failed-reposition"

    # 4) reject unknown or already loaded containers
    entry = solution.manifest.get(scanned_id)
    if entry is None or scanned_id in solution.loaded_container_ids:
        solution.conveyor_move_until_sensor(direction=solution.config.reject_move_direction)
        print(f"Rejected container {scanned_id} — waiting for container to arrive at end sensor...")
        while int(solution.conveyor_state().get("END_SENSOR", 1)) != 0:
            time.sleep(0.5)
        print("Container at end sensor. Please remove it.")
        while int(solution.conveyor_state().get("END_SENSOR", 0)) == 0:
            time.sleep(0.5)
        print("Container removed.")
        return f"rejected-{scanned_id}"

    # 5) known manifest item: accept to pickup zone
    solution.conveyor_move_until_sensor(direction=solution.config.accept_move_direction)

    row, col = solution.slot_id_to_row_col(entry.slot_id)
    if solution._is_ship_slot_accessible(row, col):
        solution._accept_from_conveyor_to_ship(entry)
        print(f"Loaded container {scanned_id} directly to ship slot {entry.slot_id}")
        solution.show_ascii_state()

        cascade_moved = solution._cascade_wharf_to_ship(on_each_move=solution.show_ascii_state)
        if cascade_moved:
            return f"loaded-{scanned_id};cascade={cascade_moved}"
        return f"loaded-{scanned_id}"

    solution._accept_from_conveyor_to_wharf(entry)
    print(f"Temporarily stored container {scanned_id} on wharf")
    solution.show_ascii_state()
    return f"wharf-stored-{scanned_id}"


def run_forever(solution, max_cycles=None):
    cycle = 0
    while True:
        result = process_once(solution)
        print(f"cycle={cycle} result={result}")

        cycle += 1
        if max_cycles is not None and cycle >= max_cycles:
            break
        time.sleep(solution.config.poll_interval_s)

## 5. Continuous Runtime Loop

In [11]:
# Single cycle:
print(process_once(solution))

# Demo run for N cycles:
run_forever(solution, max_cycles=20)

idle-no-container
cycle=0 result=idle-no-container
cycle=1 result=idle-no-container
cycle=2 result=idle-no-container
cycle=3 result=idle-no-container
cycle=4 result=idle-no-container
Rejected container 5
cycle=5 result=rejected-5
cycle=6 result=idle-no-container
cycle=7 result=idle-no-container
Rejected container 5
cycle=8 result=rejected-5
cycle=9 result=idle-no-container
cycle=10 result=idle-no-container
No ID detected. Please reposition container and wait for rescan. Retrying in 10 seconds
cycle=11 result=scan-failed-reposition
cycle=12 result=idle-no-container
cycle=13 result=idle-no-container
cycle=14 result=idle-no-container
cycle=15 result=idle-no-container
cycle=16 result=idle-no-container
cycle=17 result=idle-no-container
cycle=18 result=idle-no-container
cycle=19 result=idle-no-container


## 6. Inspect Current State (SQLite + Ship Telemetry)

In [ ]:
solution.show_ascii_state()

## 7. Clean Shutdown

In [ ]:
solution.close()
print("Disconnected MQTT client.")